# nb9.5 — Feedback Ledger: A Structured Record of Conference Comments

*Week 9, Chapter 9.5. Companion notebook.*

You will walk out of the symposium with twenty comments and the inability to remember any of them by
Sunday night. That is not a memory failure; it is a system failure. A talk lasts fifteen minutes and
the Q&A maybe ten, and during those twenty-five minutes you cannot also be taking notes. The next
day, every comment that was a vague feeling — "someone said something about the standard errors,
maybe?" — collapses into noise. Two weeks later, when you sit down to revise the paper for the
journal, the most actionable feedback is the one you cannot reconstruct.

Leah faced exactly this last summer. She presented her patent-text-novelty paper at the symposium,
got a strong set of comments, walked out feeling energized — and then could not, three days later,
list more than four of the comments. The fifth, sixth, and seventh comments were the ones that
mattered most. She knew she had heard them; she could not remember what they said.

The fix is a **feedback ledger**: a structured pandas DataFrame, written *during* and *immediately
after* the symposium, that captures every comment with five fields:

1. **comment** — what the person said, in their words.
2. **source** — who said it (discussant, audience member, advisor, peer).
3. **severity** — `critical` (must address before resubmission), `important` (should address),
   `minor` (could address).
4. **action** — the specific thing you will do about it.
5. **revision_cost_hrs** — how many hours of work the action takes.

Plus a few derived columns: status, due date, the chapter or section of the paper it touches. The
ledger is not a substitute for thinking; it is the *artifact that lets thinking happen* two weeks
later when the emotional charge has dissipated and only the structured record remains.

This notebook builds the ledger from scratch, populates it with synthetic comments from a realistic
symposium scenario, demonstrates the analyses you can run on it (severity distribution, total cost,
prioritized work plan), and exports it to Markdown for your revision packet.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import os
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta

rng = np.random.default_rng(42)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

OUTDIR = tempfile.mkdtemp(prefix="nb95_")
print("scratch output dir:", OUTDIR)
print("numpy ", np.__version__)
print("pandas", pd.__version__)

## 1. The schema: what fields a comment needs

Before we add comments, we name the columns the ledger must carry. The schema below is the *minimum*
that lets the ledger do the three jobs that justify its existence — remembering, prioritizing, and
tracking action — plus a few small additions that pay off in the revision phase.

The discipline here is that *every comment gets every field*. You cannot save a comment without
deciding its severity and action. The first time you try, this feels heavy; by the third comment,
it is automatic. The reason it works: the act of *naming* the severity forces you to think about
the comment instead of just recording it.

In [ ]:
SCHEMA = {
    "id":               "string id like 'C001'",
    "timestamp":        "when you heard it (HH:MM during talk or 'post-talk')",
    "source":           "who said it (discussant / faculty / peer / advisor / random_attendee)",
    "comment":          "the verbatim or near-verbatim quote",
    "topic":            "what part of the paper it touches (intro / data / design / table / fig / robustness / limits)",
    "severity":         "critical / important / minor",
    "action":           "the specific revision the comment calls for",
    "revision_cost_hrs":"hours of work the action takes",
    "due_date":         "when the action must be done (relative to today)",
    "status":           "open / in_progress / done / wontfix",
    "notes":            "any followup, e.g. references they mentioned",
}

print("FEEDBACK LEDGER SCHEMA")
print("=" * 78)
for k, v in SCHEMA.items():
    print(f"  {k:20} : {v}")

## 2. The empty ledger: a typed DataFrame from the start

We create the ledger as a typed pandas DataFrame so columns have the right dtypes from the start.
That matters in week three when you start computing things — without typed dates, the "due in five
days" filter does not work; without typed numeric costs, the total-hours calculation is wrong.

A common bug in feedback ledgers built ad-hoc in a notes app: severity is "critical" or "CRITICAL"
or "Critical" or "CRIT", depending on which keystroke the person hit, and grouping by severity
produces four buckets instead of one. We enforce a categorical type with fixed levels so this cannot
happen.

In [ ]:
SEVERITY_LEVELS = ["critical", "important", "minor"]
STATUS_LEVELS = ["open", "in_progress", "done", "wontfix"]

def empty_ledger():
    df = pd.DataFrame({
        "id":               pd.Series(dtype="string"),
        "timestamp":        pd.Series(dtype="string"),
        "source":           pd.Series(dtype="string"),
        "comment":          pd.Series(dtype="string"),
        "topic":            pd.Series(dtype="string"),
        "severity":         pd.Categorical([], categories=SEVERITY_LEVELS, ordered=True),
        "action":           pd.Series(dtype="string"),
        "revision_cost_hrs":pd.Series(dtype="float64"),
        "due_date":         pd.Series(dtype="object"),  # date objects
        "status":           pd.Categorical([], categories=STATUS_LEVELS, ordered=False),
        "notes":            pd.Series(dtype="string"),
    })
    return df

ledger = empty_ledger()
print(f"empty ledger has columns: {list(ledger.columns)}")
print(f"row count: {len(ledger)}")

## 3. The add-comment function: enforcing the schema

The function below takes a comment record (as a dict) and validates it against the schema before
appending. The validation is what catches the most common mistake — saving a comment without an
action. If you cannot describe the action in one sentence the moment you hear the comment, you
either did not understand the comment, or the comment is too vague to act on (and you should ask
the speaker for clarification). Either way, the function will not let you save.

In [ ]:
def add_comment(ledger, record):
    # Validate required fields.
    required = {"source", "comment", "topic", "severity", "action", "revision_cost_hrs"}
    missing = required - record.keys()
    if missing:
        raise ValueError(f"missing required fields: {missing}")
    if record["severity"] not in SEVERITY_LEVELS:
        raise ValueError(f"severity must be one of {SEVERITY_LEVELS}, got {record['severity']!r}")
    if not isinstance(record["comment"], str) or len(record["comment"]) < 5:
        raise ValueError(f"comment too short or not a string: {record['comment']!r}")
    if not isinstance(record["action"], str) or len(record["action"]) < 5:
        raise ValueError(f"action too short or not a string: {record['action']!r}")
    if not isinstance(record["revision_cost_hrs"], (int, float)) or record["revision_cost_hrs"] < 0:
        raise ValueError("revision_cost_hrs must be a non-negative number")
    # Default fields.
    record = {**record}  # copy
    record.setdefault("id", f"C{len(ledger) + 1:03d}")
    record.setdefault("timestamp", "post-talk")
    record.setdefault("status", "open")
    record.setdefault("due_date", date.today() + timedelta(days=14))
    record.setdefault("notes", "")
    new_row = pd.DataFrame([record])
    new_row["severity"] = pd.Categorical(new_row["severity"], categories=SEVERITY_LEVELS, ordered=True)
    new_row["status"] = pd.Categorical(new_row["status"], categories=STATUS_LEVELS)
    out = pd.concat([ledger, new_row], ignore_index=True)
    # Preserve dtypes.
    out["severity"] = pd.Categorical(out["severity"], categories=SEVERITY_LEVELS, ordered=True)
    out["status"] = pd.Categorical(out["status"], categories=STATUS_LEVELS)
    return out

# Test the validation: try to add a bad comment.
try:
    add_comment(ledger, {
        "source": "discussant",
        "comment": "x",  # too short
        "topic": "data",
        "severity": "critical",
        "action": "fix it",
        "revision_cost_hrs": 1.0,
    })
except ValueError as e:
    print(f"caught expected validation error: {e}")

The validation fires when a comment is too short — which is exactly the case where you almost saved
something useless and would have regretted it on day fourteen. Now we populate the ledger with a
realistic set of symposium comments from Leah's patent-text-novelty talk.

## 4. Populating the ledger: Leah's symposium comments

Below are the ten comments Leah captures during and after her talk. They are realistic: a mix of
methodological ("did you cluster by firm?"), framing ("the contribution sentence isn't sharp"),
data-construction ("how do you handle continuation patents?"), and forward-looking ("have you tried
the same idea on trademarks?"). Each gets a severity tag (was it a *show-stopper* for the journal,
or a *nice-to-have*?), an action, and a cost estimate.

In [ ]:
COMMENTS = [
    {
        "timestamp": "post-talk",
        "source":    "discussant",
        "comment":   "Your novelty score uses cosine similarity to the prior-art corpus, but the "
                     "corpus is incomplete before 2001. Patents filed before 2001 have artificially "
                     "high novelty scores by construction.",
        "topic":     "data",
        "severity":  "critical",
        "action":    "Re-construct novelty score using only post-2001 sample; report the headline "
                     "with and without pre-2001 patents in the corpus.",
        "revision_cost_hrs": 6.0,
        "notes":     "References: Hall et al. (2001) NBER patent database; check Younge & Kuhn (2016).",
    },
    {
        "timestamp": "during Q&A",
        "source":    "faculty",
        "comment":   "You cluster standard errors by year. Why not by firm? Firm-level clustering "
                     "would be more conservative given correlation in patents from the same firm.",
        "topic":     "design",
        "severity":  "important",
        "action":    "Re-estimate the headline with two-way clustering (firm and year). Report both "
                     "SEs in the table footnote.",
        "revision_cost_hrs": 1.5,
        "notes":     "",
    },
    {
        "timestamp": "during Q&A",
        "source":    "peer",
        "comment":   "How do you handle continuation patents? They cite the parent patent verbatim, "
                     "which would inflate or deflate novelty depending on direction.",
        "topic":     "data",
        "severity":  "critical",
        "action":    "Identify continuation patents from the application data; drop or flag them; "
                     "report the result both ways.",
        "revision_cost_hrs": 4.0,
        "notes":     "PatentsView has a 'continuation' flag; verify.",
    },
    {
        "timestamp": "post-talk",
        "source":    "advisor",
        "comment":   "Your contribution sentence is buried in paragraph 3. Move it to the first "
                     "paragraph and sharpen it to one sentence.",
        "topic":     "intro",
        "severity":  "important",
        "action":    "Rewrite intro paragraph 1 with contribution as the second sentence. Cut "
                     "paragraph 3.",
        "revision_cost_hrs": 1.0,
        "notes":     "",
    },
    {
        "timestamp": "post-talk",
        "source":    "discussant",
        "comment":   "Your headline coefficient is identified off variation within firm-year cells. "
                     "Add a column showing the result with firm-by-year fixed effects to see how "
                     "much within-cell variation drives it.",
        "topic":     "table",
        "severity":  "important",
        "action":    "Add column (5) to Table 1 with firm-year fixed effects; comment in text on "
                     "whether the headline survives.",
        "revision_cost_hrs": 2.0,
        "notes":     "",
    },
    {
        "timestamp": "during talk",
        "source":    "random_attendee",
        "comment":   "Your slide 8 says 'see appendix' for the placebo test but I couldn't find the "
                     "appendix table.",
        "topic":     "fig",
        "severity":  "minor",
        "action":    "Make sure Appendix Table A3 is referenced and numbered in the manuscript "
                     "package.",
        "revision_cost_hrs": 0.5,
        "notes":     "",
    },
    {
        "timestamp": "during Q&A",
        "source":    "faculty",
        "comment":   "Your sample starts in 2001. What happens if you start earlier? The patent "
                     "data goes back to 1976 in PatentsView.",
        "topic":     "robustness",
        "severity":  "minor",
        "action":    "Add appendix table extending sample back to 1976; flag the data-quality "
                     "caveat about pre-2001 prior art.",
        "revision_cost_hrs": 3.0,
        "notes":     "Tied to the C001 critical comment.",
    },
    {
        "timestamp": "post-talk",
        "source":    "advisor",
        "comment":   "The forest plot in figure 3 has 12 categories. Bin them down to 6 industries.",
        "topic":     "fig",
        "severity":  "minor",
        "action":    "Re-aggregate industries to SIC 2-digit; redraw forest plot with 6 bars.",
        "revision_cost_hrs": 1.0,
        "notes":     "",
    },
    {
        "timestamp": "during Q&A",
        "source":    "discussant",
        "comment":   "Have you tried the same novelty measure on trademarks? It would be a natural "
                     "external-validity test.",
        "topic":     "limits",
        "severity":  "minor",
        "action":    "Acknowledge in the conclusion as future work; do not run for this submission "
                     "(out of scope).",
        "revision_cost_hrs": 0.2,
        "notes":     "Flagged in 'future work' paragraph.",
    },
    {
        "timestamp": "post-talk",
        "source":    "peer",
        "comment":   "Your table 1 standard errors are reported to 4 decimals. That's false precision.",
        "topic":     "table",
        "severity":  "minor",
        "action":    "Round all SEs in all tables to 3 decimals; apply uniformly.",
        "revision_cost_hrs": 0.2,
        "notes":     "",
    },
]

for c in COMMENTS:
    ledger = add_comment(ledger, c)

print(f"ledger now has {len(ledger)} comments")
print()
print(ledger[["id", "source", "severity", "topic", "revision_cost_hrs"]].to_string(index=False))

## 5. The first analysis: severity distribution and total cost

The first question to ask the ledger is: *how much work am I in for?* Sum the cost column by
severity, and you have an answer. The second question is: *which of these is non-negotiable?* Count
the critical comments. The ledger answers both immediately.

In [ ]:
summary = ledger.groupby("severity", observed=True).agg(
    count=("id", "count"),
    total_hours=("revision_cost_hrs", "sum"),
).reset_index()
print("By severity:")
print(summary.to_string(index=False))

total_hrs = ledger["revision_cost_hrs"].sum()
critical_hrs = ledger.loc[ledger["severity"] == "critical", "revision_cost_hrs"].sum()
print(f"\nTotal revision cost   : {total_hrs:.1f} hours")
print(f"Critical-only subset  : {critical_hrs:.1f} hours")
print(f"  -> at 4 hours per evening, critical subset = {critical_hrs / 4:.1f} evenings of work.")

## 6. The second analysis: where in the paper is the feedback concentrated?

Group by topic. If three of Leah's ten comments touch the data construction, that is a signal — the
data section is the weakest part of the paper, and a careful revision starts there. If most comments
are minor cosmetic table issues, the paper is solid and Leah is mostly polishing.

In [ ]:
by_topic = (
    ledger.groupby("topic", observed=True)
          .agg(count=("id", "count"),
               total_hours=("revision_cost_hrs", "sum"),
               critical=("severity", lambda s: (s == "critical").sum()))
          .sort_values("count", ascending=False)
)
print("By topic:")
print(by_topic.to_string())

Two topics (data, table, fig) carry most of the comments, and data carries both critical ones. The
revision plan writes itself: spend the first two days entirely on data (continuation patents,
pre-2001 corpus), and only after that move to the table and figure fixes.

## 7. The prioritized work plan

A prioritized plan sorts the ledger by severity (critical → important → minor) and within each
severity by *expected value per hour* — the ratio of severity weight to cost. The work plan is then
the cumulative-hours column, which lets Leah read off "the first 12 hours of work cover all the
critical comments and three of the four important ones," which is the kind of concrete sentence she
can carry into a planning meeting with her advisor.

In [ ]:
SEV_WEIGHT = {"critical": 3.0, "important": 1.0, "minor": 0.3}
# Make a workable copy.
plan = ledger.copy()
plan["sev_weight"] = plan["severity"].astype(str).map(SEV_WEIGHT)
plan["score"] = plan["sev_weight"] / plan["revision_cost_hrs"].clip(lower=0.2)

plan["sev_rank"] = plan["severity"].cat.codes  # 0=critical,1=important,2=minor
plan = plan.sort_values(["sev_rank", "score"], ascending=[True, False]).reset_index(drop=True)
plan["cum_hours"] = plan["revision_cost_hrs"].cumsum()

cols = ["id", "severity", "topic", "revision_cost_hrs", "score", "cum_hours", "action"]
print("Prioritized revision plan:")
with pd.option_context("display.max_colwidth", 60, "display.width", 200):
    print(plan[cols].round(2).to_string(index=False))

## 8. The third analysis: status tracking over time

The ledger is most useful in the *weeks after* the symposium, when Leah starts working through the
plan. She updates `status` from `open` to `in_progress` to `done` as she finishes each item. The
ledger then becomes a Gantt-style view of her revision progress — and, crucially, an exhibit she can
attach to the resubmission letter as evidence she addressed every comment from the symposium.

In [ ]:
# Simulate two weeks of progress: Leah finishes some items.
def update_status(ledger, comment_id, new_status):
    out = ledger.copy()
    out.loc[out["id"] == comment_id, "status"] = new_status
    out["status"] = pd.Categorical(out["status"], categories=STATUS_LEVELS)
    return out

# Day 3: she finishes the minor stuff first (warming up).
ledger = update_status(ledger, "C006", "done")  # appendix reference fix
ledger = update_status(ledger, "C010", "done")  # SE rounding
# Day 5: starts the critical data work.
ledger = update_status(ledger, "C001", "in_progress")
ledger = update_status(ledger, "C003", "in_progress")
# Day 10: data work done.
ledger = update_status(ledger, "C001", "done")
ledger = update_status(ledger, "C003", "done")
# Day 12: cluster SE done.
ledger = update_status(ledger, "C002", "done")

# Status snapshot.
status_summary = (
    ledger.groupby("status", observed=True)
          .agg(count=("id", "count"),
               total_hours=("revision_cost_hrs", "sum"))
)
print("Status snapshot (after 2 weeks of revision):")
print(status_summary.to_string())

done_hrs = ledger.loc[ledger["status"] == "done", "revision_cost_hrs"].sum()
remaining_hrs = ledger.loc[ledger["status"] != "done", "revision_cost_hrs"].sum()
print(f"\nDone     : {done_hrs:.1f} hours  ({done_hrs / total_hrs * 100:.0f}% by cost)")
print(f"Remaining: {remaining_hrs:.1f} hours")
critical_remaining = ledger.loc[
    (ledger["severity"] == "critical") & (ledger["status"] != "done"),
    "id",
].tolist()
print(f"Critical still open: {critical_remaining}  -- empty list means 'safe to resubmit'.")

## 9. Visualization: the burn-down chart

A burn-down chart shows remaining work over time — the kind of plot project managers live by.
Below we draw a simple version: a stacked bar of remaining cost by severity at three snapshots. The
visual signature you want: the **critical bar** shrinks first, then important, then minor. If the
critical bar is the *last* to shrink, you are working in the wrong order.

In [ ]:
# Three snapshots: day 0 (everything open), day 5 (some progress), day 14 (most critical done).
def remaining_by_severity_at(snapshot_done_ids, ledger):
    out = ledger.copy()
    out["status"] = pd.Categorical(
        ["done" if cid in snapshot_done_ids else "open" for cid in out["id"]],
        categories=STATUS_LEVELS,
    )
    rem = out.loc[out["status"] != "done"].groupby("severity", observed=True)["revision_cost_hrs"].sum()
    return rem.reindex(SEVERITY_LEVELS).fillna(0.0)

snap_0 = remaining_by_severity_at(set(), ledger)
snap_5 = remaining_by_severity_at({"C006", "C010"}, ledger)
snap_14 = remaining_by_severity_at({"C001", "C002", "C003", "C006", "C010"}, ledger)

xs = ["Day 0", "Day 5", "Day 14"]
crit = [snap_0["critical"], snap_5["critical"], snap_14["critical"]]
imp = [snap_0["important"], snap_5["important"], snap_14["important"]]
min_ = [snap_0["minor"], snap_5["minor"], snap_14["minor"]]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(xs, crit, label="critical", color="#c1272d")
ax.bar(xs, imp, bottom=crit, label="important", color="#e09e3c")
ax.bar(xs, min_, bottom=np.array(crit) + np.array(imp), label="minor", color="#9bb29b")
ax.set_ylabel("Remaining revision cost (hours)")
ax.set_title("Revision burn-down: hours by severity over time")
ax.legend(loc="upper right")
fig.tight_layout()
burn_path = os.path.join(OUTDIR, "burn_down.png")
fig.savefig(burn_path, dpi=150)
plt.close(fig)
print(f"burn-down chart saved to {burn_path}")
print(f"\nat day 14: critical = {snap_14['critical']:.1f} h  important = {snap_14['important']:.1f} h  minor = {snap_14['minor']:.1f} h")

## 10. Exporting the ledger to Markdown for the revision packet

The ledger lives in a notebook while you build it; in the week you submit the revision, you want it
as a plain-Markdown file the journal editor can read. The export below produces a one-row-per-comment
revision memo that you attach to the resubmission letter.

In [ ]:
def export_ledger_markdown(ledger, out_path):
    lines = ["# Revision Memo: response to symposium feedback\n"]
    lines.append(f"_Generated: {date.today().isoformat()}_\n")
    lines.append(f"_Total comments: {len(ledger)} | Critical: "
                 f"{(ledger['severity'] == 'critical').sum()} | "
                 f"Done: {(ledger['status'] == 'done').sum()}_\n")
    lines.append("## Summary table\n")
    summary_view = ledger[["id", "source", "severity", "topic", "status", "revision_cost_hrs"]].copy()
    lines.append(summary_view.to_markdown(index=False))
    lines.append("\n\n## Comment-by-comment response\n")
    for _, row in ledger.iterrows():
        lines.append(f"### {row['id']} — {row['topic']} | {row['severity']} | "
                     f"status: {row['status']}\n")
        lines.append(f"**Source:** {row['source']}\n")
        lines.append(f"**Comment:** {row['comment']}\n")
        lines.append(f"**Action:** {row['action']}\n")
        lines.append(f"**Cost:** {row['revision_cost_hrs']} hours")
        if row["notes"]:
            lines.append(f"\n**Notes:** {row['notes']}")
        lines.append("\n")
    return "\n".join(lines)

memo_path = os.path.join(OUTDIR, "revision_memo.md")
memo = export_ledger_markdown(ledger, memo_path)
with open(memo_path, "w") as fh:
    fh.write(memo)
print(f"revision memo written to {memo_path} ({len(memo)} chars)")
print()
print("--- first 800 chars ---")
print(memo[:800])

## 11. The point of the ledger

A symposium that produced ten unstructured comments produces zero revisions a month later. A
symposium that produced ten *ledger entries* produces nine revisions a month later, in priority
order, with a written record of which you addressed and why. The difference is not memory or
willpower; it is *structure*. The ledger is small — eleven columns, a pandas DataFrame, fifty lines
of code — and it is the single highest-leverage artifact you build during the symposium week.

Two habits to carry forward:

- **Capture during, not after.** The instinct is to listen now and write later. By the third
  comment you have lost the first one. Carry a notebook (paper or digital) into the talk; let your
  co-author or a friend transcribe the Q&A; commit comments to the ledger within an hour of the
  talk ending.
- **Severity is a choice you make once, in the moment.** Re-deciding severity two weeks later is a
  recipe for downgrading the comments you do not want to address. The first-impression severity
  tag is the honest one.

The exact conventions — when to flag a comment as `wontfix` and how to defend that to a referee, what
the response letter looks like when you cite the ledger, and how to handle a comment that turns out
to be flat-out wrong — live in the chapter prose.

---

### Your Turn

1. **Build the ledger for *your* practice talk.** Find a friend, give your 12-minute symposium talk
   to them, and have them ask 5 questions. Capture each into a ledger row with all six required
   fields. How long did the capture take, and how complete is the record an hour later? A day
   later?
2. **Stress-test the validation.** Try to add a comment with `severity = "huge"`, or with an empty
   action string, or with a negative cost. Confirm the validation fires. Then loosen the validation
   for `wontfix` status — should a `wontfix` comment require an action? (Hint: yes, "no action;
   reason: out of scope.")
3. **Compute the cost-quality frontier.** Plot total revision hours on the x-axis against
   percentage-of-critical-resolved on the y-axis, as you sweep over the prioritized order. Where
   does the curve flatten? That is your "good enough to resubmit" point.